# LLM Fine-Tuning with QLoRA - Friend Account Clean Run

This notebook fine-tunes Mistral-7B-Instruct-v0.2 using QLoRA on context-aware Turkish legal QA data.

This version is configured for a safer Colab run:
- max_steps = 300
- checkpoint every 50 steps
- fp16=False and bf16=False to avoid BF16 GradScaler errors
- outputs saved directly to Google Drive

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU yok. Runtime > Change runtime type > T4 GPU seç.")

CUDA available: True
GPU: Tesla T4


In [3]:
import os

project_path = "/content/drive/MyDrive/turkish_legal_rag"

processed_path = f"{project_path}/data/processed"
outputs_path = f"{project_path}/outputs"
metrics_path = f"{outputs_path}/metrics"
models_path = f"{outputs_path}/models"

train_jsonl_path = f"{processed_path}/llm_sft_train.jsonl"
val_jsonl_path = f"{processed_path}/llm_sft_val.jsonl"
test_jsonl_path = f"{processed_path}/llm_sft_test.jsonl"

adapter_output_path = f"{models_path}/mistral_legal_qlora_adapter_300steps"

print("Project exists:", os.path.exists(project_path))
print("Train exists:", os.path.exists(train_jsonl_path))
print("Val exists:", os.path.exists(val_jsonl_path))
print("Test exists:", os.path.exists(test_jsonl_path))
print("Adapter output:", adapter_output_path)

Project exists: True
Train exists: True
Val exists: True
Test exists: True
Adapter output: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps


In [4]:
for path in [train_jsonl_path, val_jsonl_path, test_jsonl_path]:
    print(path)
    print("Exists:", os.path.exists(path))
    if os.path.exists(path):
        print("Size MB:", round(os.path.getsize(path) / (1024 * 1024), 2))
    print("-" * 80)

/content/drive/MyDrive/turkish_legal_rag/data/processed/llm_sft_train.jsonl
Exists: True
Size MB: 32.03
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/llm_sft_val.jsonl
Exists: True
Size MB: 4.03
--------------------------------------------------------------------------------
/content/drive/MyDrive/turkish_legal_rag/data/processed/llm_sft_test.jsonl
Exists: True
Size MB: 4.02
--------------------------------------------------------------------------------


In [5]:
!pip install -q -U transformers accelerate peft trl bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 42.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.7 MB/s eta 0:00:00


In [6]:
import os
import json
import gc
import pandas as pd
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [7]:
data_files = {
    "train": train_jsonl_path,
    "validation": val_jsonl_path
}

dataset = load_dataset("json", data_files=data_files)

print(dataset)
print(dataset["train"][0].keys())
print(dataset["train"][0]["text"][:1200])

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'question', 'answer', 'source', 'score'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['text', 'question', 'answer', 'source', 'score'],
        num_rows: 500
    })
})
dict_keys(['text', 'question', 'answer', 'source', 'score'])
<s>[INST] Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.
Cevabı yalnızca verilen bağlama göre üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, net ve Türkçe olmalı.

Bağlam:
ÜÇÜNCÜ KISIM

Yaptırımlar

İKİNCİ BÖLÜM

Güvenlik Tedbirleri

 

Belli hakları kullanmaktan yoksun bırakılma

MADDE 53. - (1) Kişi, kasten işlemiş olduğu suçtan dolayı hapis cezasına mahkûmiyetin kanuni sonucu olarak;

a) Sürekli, süreli veya geçici bir kamu görevinin üstlenilmesinden; bu kapsamda, Türkiye Büyük Millet Meclisi üyeliğinden veya Devlet, il, belediye, köy veya bunların denetim ve gözetimi altında bulunan kurum ve kuruluşlarca verilen, atamaya 

In [8]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
model_name = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)

config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.10k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Tokenizer loaded.
Pad token: </s>


In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False

print("Model loaded:", model_name)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.1k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded: mistralai/Mistral-7B-Instruct-v0.2


In [11]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

print("Model prepared for k-bit training.")

Model prepared for k-bit training.


In [12]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("LoRA config ready.")

LoRA config ready.


In [13]:
os.makedirs(adapter_output_path, exist_ok=True)

MAX_STEPS = 300
SAVE_STEPS = 50
EVAL_STEPS = 50

sft_config = SFTConfig(
    output_dir=adapter_output_path,

    # Dataset
    dataset_text_field="text",
    max_length=512,
    packing=False,

    # Controlled run
    max_steps=MAX_STEPS,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    # Important: disable AMP/GradScaler to avoid BF16 error
    fp16=False,
    bf16=False,

    # Memory/performance
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",

    # Logging / eval / save
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,

    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=6,

    report_to="none"
)

print("Safe checkpointed SFT config ready.")
print("Max steps:", MAX_STEPS)
print("Save every:", SAVE_STEPS)
print("Eval every:", EVAL_STEPS)
print("Output:", adapter_output_path)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Safe checkpointed SFT config ready.
Max steps: 300
Save every: 50
Eval every: 50
Output: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps


In [14]:
try:
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=lora_config,
        processing_class=tokenizer
    )
except TypeError:
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        peft_config=lora_config,
        tokenizer=tokenizer
    )

print("Trainer ready.")

Adding EOS to train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/500 [00:00<?, ? examples/s]

Trainer ready.


In [15]:
train_result = trainer.train()

print("Training finished.")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.685523,0.606924,0.679087,204800.000000,0.846215
100,0.269569,0.320998,0.329308,409600.000000,0.915593
150,0.168477,0.135973,0.172629,614400.000000,0.962164
200,0.059790,0.062688,0.093731,819200.000000,0.981538
250,0.034563,0.038844,0.064242,1024000.000000,0.988372


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.685523,0.606924,0.679087,204800.000000,0.846215
100,0.269569,0.320998,0.329308,409600.000000,0.915593
150,0.168477,0.135973,0.172629,614400.000000,0.962164
200,0.059790,0.062688,0.093731,819200.000000,0.981538
250,0.034563,0.038844,0.064242,1024000.000000,0.988372
300,0.030256,0.036741,0.062324,1228800.000000,0.989029


Training finished.


In [16]:
print("Adapter output exists:", os.path.exists(adapter_output_path))
print("Adapter output path:", adapter_output_path)

for root, dirs, files in os.walk(adapter_output_path):
    level = root.replace(adapter_output_path, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files:
        print(f"{indent}  {file}")

Adapter output exists: True
Adapter output path: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps
mistral_legal_qlora_adapter_300steps/
  README.md
  checkpoint-50/
    README.md
    adapter_model.safetensors
    adapter_config.json
    chat_template.jinja
    tokenizer_config.json
    tokenizer.json
    training_args.bin
    optimizer.pt
    scheduler.pt
    rng_state.pth
    trainer_state.json
  checkpoint-100/
    README.md
    adapter_model.safetensors
    adapter_config.json
    chat_template.jinja
    tokenizer_config.json
    tokenizer.json
    training_args.bin
    optimizer.pt
    scheduler.pt
    rng_state.pth
    trainer_state.json
  checkpoint-150/
    README.md
    adapter_model.safetensors
    adapter_config.json
    chat_template.jinja
    tokenizer_config.json
    tokenizer.json
    training_args.bin
    optimizer.pt
    scheduler.pt
    rng_state.pth
    trainer_state.json
  checkpoint-200/
    README.md
    adapter_model.saf

In [17]:
trainer.save_model(adapter_output_path)
tokenizer.save_pretrained(adapter_output_path)

print("Final adapter saved to:", adapter_output_path)

print("\nFiles in adapter folder:")
for item in os.listdir(adapter_output_path):
    print("-", item)

Final adapter saved to: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps

Files in adapter folder:
- README.md
- checkpoint-50
- checkpoint-100
- checkpoint-150
- checkpoint-200
- checkpoint-250
- checkpoint-300
- adapter_model.safetensors
- adapter_config.json
- chat_template.jinja
- tokenizer_config.json
- tokenizer.json
- training_args.bin


In [18]:
required_files = [
    "adapter_config.json",
    "adapter_model.safetensors"
]

print("Checking required adapter files...")

for file in required_files:
    file_path = os.path.join(adapter_output_path, file)
    print(file, "exists:", os.path.exists(file_path))
    if os.path.exists(file_path):
        print("Size MB:", round(os.path.getsize(file_path) / (1024 * 1024), 2))

Checking required adapter files...
adapter_config.json exists: True
Size MB: 0.0
adapter_model.safetensors exists: True
Size MB: 80.06


In [19]:
log_history = trainer.state.log_history

log_df = pd.DataFrame(log_history)

display(log_df.tail(20))

log_path = f"{metrics_path}/mistral_legal_qlora_300steps_training_log.csv"

log_df.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

print("Training log saved:", log_path)

,loss,grad_norm,learning_rate,entropy,num_tokens,mean_token_accuracy,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,eval_entropy,eval_num_tokens,eval_mean_token_accuracy,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
17,NaN,NaN,NaN,NaN,NaN,NaN,0.30,150,0.135973,459.0901,1.089,1.089,0.172629,614400.0,0.962164,NaN,NaN,NaN,NaN,NaN
18,0.142805,1.296875,9.514378e-05,0.179080,655360.0,0.960250,0.32,160,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
19,0.091703,1.031250,8.440987e-05,0.131353,696320.0,0.973263,0.34,170,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,0.110084,1.132812,7.385749e-05,0.148200,737280.0,0.970279,0.36,180,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
21,0.080132,0.714844,6.360951e-05,0.126642,778240.0,0.977202,0.38,190,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
22,0.059790,0.863281,5.378524e-05,0.087108,819200.0,0.982534,0.40,200,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
23,NaN,NaN,NaN,NaN,NaN,NaN,0.40,200,0.062688,459.5721,1.088,1.088,0.093731,819200.0,0.981538,NaN,NaN,NaN,NaN,NaN
24,0.049779,0.539062,4.449909e-05,0.086177,860160.0,0.984883,0.42,210,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,0.051085,0.640625,3.585918e-05,0.074779,901120.0,0.984051,0.44,220,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,0.041487,0.419922,2.796610e-05,0.072305,942080.0,0.986913,0.46,230,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Training log saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_300steps_training_log.csv


In [20]:
llm_finetune_summary_df = pd.DataFrame([{
    "base_model": model_name,
    "method": "QLoRA",
    "train_file": train_jsonl_path,
    "val_file": val_jsonl_path,
    "train_rows": len(dataset["train"]),
    "val_rows": len(dataset["validation"]),
    "max_steps": MAX_STEPS,
    "save_steps": SAVE_STEPS,
    "eval_steps": EVAL_STEPS,
    "per_device_train_batch_size": sft_config.per_device_train_batch_size,
    "per_device_eval_batch_size": sft_config.per_device_eval_batch_size,
    "gradient_accumulation_steps": sft_config.gradient_accumulation_steps,
    "learning_rate": sft_config.learning_rate,
    "max_length": sft_config.max_length,
    "fp16": sft_config.fp16,
    "bf16": sft_config.bf16,
    "lora_r": lora_config.r,
    "lora_alpha": lora_config.lora_alpha,
    "lora_dropout": lora_config.lora_dropout,
    "adapter_output_path": adapter_output_path
}])

display(llm_finetune_summary_df)

summary_path = f"{metrics_path}/mistral_legal_qlora_300steps_summary.csv"

llm_finetune_summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig"
)

print("Summary saved:", summary_path)

,base_model,method,train_file,val_file,train_rows,val_rows,max_steps,save_steps,eval_steps,per_device_train_batch_size,per_device_eval_batch_size,gradient_accumulation_steps,learning_rate,max_length,fp16,bf16,lora_r,lora_alpha,lora_dropout,adapter_output_path
0,mistralai/Mistral-7B-Instruct-v0.2,QLoRA,/content/drive/MyDrive/turkish_legal_rag/data/...,/content/drive/MyDrive/turkish_legal_rag/data/...,4000,500,300,50,50,1,1,8,0.0002,512,False,False,16,32,0.05,/content/drive/MyDrive/turkish_legal_rag/outpu...


Summary saved: /content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_300steps_summary.csv


In [22]:
def generate_with_finetuned_model(prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return text

In [23]:
sample = dataset["validation"][0]

prompt_text = sample["text"].split("[/INST]")[0] + "[/INST]"

print("PROMPT:")
print(prompt_text[:1500])

generated = generate_with_finetuned_model(prompt_text)

print("\nGENERATED:")
print(generated[-1500:])

print("\nEXPECTED:")
print(sample["answer"])

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


PROMPT:
<s>[INST] Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.
Cevabı yalnızca verilen bağlama göre üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, net ve Türkçe olmalı.

Bağlam:
ÜÇÜNCÜ KISIM

CUMHURİYETİN TEMEL ORGANLARI

ÜÇÜNCÜ BÖLÜM

Yargı
…Öncesi

II. Yüksek mahkemeler

 

A. Anayasa Mahkemesi

 

1. Kuruluşu

Madde 146 – (Değişik: 7/5/2010-5982/16 md.)

Anayasa Mahkemesi onbeş üyeden kurulur. [77]

Türkiye Büyük Millet Meclisi; iki üyeyi Sayıştay Genel Kurulunun kendi başkan ve üyeleri arasından, her boş yer için gösterecekleri üçer aday içinden, bir üyeyi ise baro başkanlarının serbest avukatlar arasından gösterecekleri üç aday içinden yapacağı gizli oylamayla seçer. Türkiye Büyük Millet Meclisinde yapılacak bu seçimde, her boş üyelik için ilk oylamada üye tam sayısının üçte iki ve ikinci oylamada üye tam sayısının salt çoğunluğu aranır. İkinci oylamada salt çoğunluk sağlanamazsa, bu oylamada en çok oy alan iki aday için üçüncü oylama

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



GENERATED:
[INST] Sen Türk hukuk metinleri için çalışan dikkatli bir soru-cevap asistanısın.
Cevabı yalnızca verilen bağlama göre üret.
Bağlamda açıkça bulunmayan bilgileri uydurma.
Cevap kısa, net ve Türkçe olmalı.

Bağlam:
ÜÇÜNCÜ KISIM

CUMHURİYETİN TEMEL ORGANLARI

ÜÇÜNCÜ BÖLÜM

Yargı
…Öncesi

II. Yüksek mahkemeler

 

A. Anayasa Mahkemesi

 

1. Kuruluşu

Madde 146 – (Değişik: 7/5/2010-5982/16 md.)

Anayasa Mahkemesi onbeş üyeden kurulur. [77]

Türkiye Büyük Millet Meclisi; iki üyeyi Sayıştay Genel Kurulunun kendi başkan ve üyeleri arasından, her boş yer için gösterecekleri üçer aday içinden, bir üyeyi ise baro başkanlarının serbest avukatlar arasından gösterecekleri üç aday içinden yapacağı gizli oylamayla seçer. Türkiye Büyük Millet Meclisinde yapılacak bu seçimde, her boş üyelik için ilk oylamada üye tam sayısının üçte iki ve ikinci oylamada üye tam sayısının salt çoğunluğu aranır. İkinci oylamada salt çoğunluk sağlanamazsa, bu oylamada üye tam sayısı ile üye tam sayısının üçte

In [24]:
zip_output_path = f"{models_path}/mistral_legal_qlora_adapter_300steps.zip"

!zip -r "{zip_output_path}" "{adapter_output_path}"

print("Adapter zipped to:", zip_output_path)
print("Zip exists:", os.path.exists(zip_output_path))

if os.path.exists(zip_output_path):
    print("Zip size MB:", round(os.path.getsize(zip_output_path) / (1024 * 1024), 2))

  adding: content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps/ (stored 0%)
  adding: content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps/README.md (deflated 45%)
  adding: content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps/checkpoint-50/ (stored 0%)
  adding: content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps/checkpoint-50/README.md (deflated 65%)
  adding: content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps/checkpoint-50/adapter_model.safetensors (deflated 21%)
  adding: content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps/checkpoint-50/adapter_config.json (deflated 59%)
  adding: content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps/checkpoint-50/chat_template.jinja (deflated 64%)
  adding: content/drive/MyDri

In [25]:
print("FINAL CHECK")
print("=" * 80)

print("Adapter folder:", adapter_output_path)
print("Adapter folder exists:", os.path.exists(adapter_output_path))

adapter_config_path = os.path.join(adapter_output_path, "adapter_config.json")
adapter_model_path = os.path.join(adapter_output_path, "adapter_model.safetensors")

print("adapter_config.json:", os.path.exists(adapter_config_path))
print("adapter_model.safetensors:", os.path.exists(adapter_model_path))

if os.path.exists(adapter_model_path):
    print("adapter_model.safetensors size MB:", round(os.path.getsize(adapter_model_path) / (1024 * 1024), 2))

print("\nMetrics files:")
for file in [
    f"{metrics_path}/mistral_legal_qlora_300steps_training_log.csv",
    f"{metrics_path}/mistral_legal_qlora_300steps_summary.csv"
]:
    print(file, os.path.exists(file))

print("\nDone.")

FINAL CHECK
Adapter folder: /content/drive/MyDrive/turkish_legal_rag/outputs/models/mistral_legal_qlora_adapter_300steps
Adapter folder exists: True
adapter_config.json: True
adapter_model.safetensors: True
adapter_model.safetensors size MB: 80.06

Metrics files:
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_300steps_training_log.csv True
/content/drive/MyDrive/turkish_legal_rag/outputs/metrics/mistral_legal_qlora_300steps_summary.csv True

Done.


In [27]:
def generate_answer_only(prompt, max_new_tokens=180):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(model.device)

    input_length = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = outputs[0][input_length:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True)

    return answer.strip()

In [28]:
sample = dataset["validation"][0]

prompt_text = sample["text"].split("[/INST]")[0] + "[/INST]"

generated_answer = generate_answer_only(prompt_text)

print("GENERATED ANSWER:")
print(generated_answer)

print("\nEXPECTED ANSWER:")
print(sample["answer"])

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


GENERATED ANSWER:
üye tam sayısı ile üye tam sayısının üçte iki çoğunluğu aranır.

Millet Meclisinde yapılacak her boş üyelik için ilk oylamada tam sayısının üçte iki çoğunlukunun sağlanması hâlinde, bu çoğunluk için gerekli olan ikinci oylamaya engel olmamakla birlikte, bu çoğunluk için gerekli olan üyeyi, ilk oylamada tam sayısının üçte iki çoğunlukunun sağ

EXPECTED ANSWER:
Anayasa Mahkemesi, kanunların şekil bakımından denetlenmesinde, son oylamanın öngörülen çoğunlukla yapılıp yapılmadığını ve Anayasa değişikliklerinde teklif ve oylama çoğunluğuna ve ivedilikle görüşülemeyeceği şartına uyulup uyulmadığını inceler. Bu denetim, belirli formalitelerin yerine getirildiğinden emin olmak amacıyla yapılır.
